<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer-banner-light.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 06 — The Architect

Every previous notebook had **you** design the workflow. You picked the nodes, wrote the prompts,
wired the `depends_on` edges. This one hands that job to an agent.

The **Architect** takes a plain-English intent and produces a registered, runnable
[WorkflowPackage](03_graph_agents.ipynb) — it decides the steps, finds the tools each one needs,
writes the prompts, wires the graph, **runs what it built**, judges the output against criteria it
derived itself, and repairs anything that failed. If it can't do the job it refuses and tells you
exactly what's missing, rather than shipping a workflow that pretends.

You'll learn:

1. **The two-agent split** — why planning and building are separate calls, and why that is the
   single biggest factor in whether this works on a small model.
2. **The plan** — `plan_and_resolve`, and the capability ladder that runs *in code*.
3. **The build** — `ArchitectAgent.build`, watched step by step.
4. **The gate** — validation, and the rule that stops a workflow from accepting a parameter it
   ignores.
5. **The proof** — `derive_acceptance` + `verify_workflow`: the closed loop that runs and judges
   the workflow before it registers.
6. **The refusal** — `WorkflowInfeasible`, and why a clear blocker beats a plausible fake.

> **Runtime:** ~3–10 minutes on a local 9B model — it varies a lot, because how long §5 takes
> depends on how much the Architect has to repair. Every section prints as it goes, so you can
> watch rather than wait.

---

## 1. Setup

Same boilerplate as the earlier notebooks. The Architect is pure Python — no optional extras
needed beyond what you already have.

The validated local choice is **`qwen/qwen3.5-9b`** in LM Studio, with the local server on
(port 1234). A 9B model is a deliberate test: the Architect is designed so that *the framework*
carries the correctness, not the model's memory. If it works here, a bigger model is strictly
easier.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, json, textwrap
from pathlib import Path

# Point Python at the repo root when running from tutorials/
NB_DIR    = Path(os.getcwd())
REPO_ROOT = NB_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

# Everything the Architect writes lands here — staging while it builds, then the
# registry. `tutorials/tmp/` is gitignored, and it is worth opening afterwards:
# the graph.yaml in there is the actual deliverable.
WORK     = NB_DIR / "tmp" / "architect"
STAGING  = WORK / "staging"
REGISTRY = WORK / "registry"
WORK.mkdir(parents=True, exist_ok=True)
print(f"working dir: {WORK}")

In [ ]:
# ── LM Studio connection ──────────────────────────────────────────────────────
LM_STUDIO_URL   = "http://localhost:1234/v1"
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"
CONTEXT_WINDOW  = 32_768

from neurosurfer.llm.providers.openai import OpenAICompatProvider

provider = OpenAICompatProvider(
    model          = LM_STUDIO_MODEL,
    base_url       = LM_STUDIO_URL,
    api_key        = "lm-studio",
    context_window = CONTEXT_WINDOW,
)

# For a hosted model instead, swap the two lines above for:
#   from neurosurfer.llm.providers.openai import OpenAIProvider
#   provider = OpenAIProvider(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"])

print(f"provider ready: {LM_STUDIO_MODEL}")

In [ ]:
# A tiny progress printer. Every Architect entry point takes `notify=`, and it is
# the whole reason this notebook is watchable rather than a 6-minute blank cell.
def say(msg: str) -> None:
    print(f"  · {msg}")

---

## 2. What the Architect actually is

Two agents, not one — and that split is the whole design.

```
      your intent
           │
           ▼
   ┌───────────────┐   ONE structured call. No tools, no loop.
   │    PLANNER    │   "What are the steps, and which reach outside the model?"
   └───────┬───────┘
           │  WorkflowPlan
           ▼
   ┌───────────────┐   Runs IN CODE, not by asking the model:
   │   CAPABILITY  │   catalog search → MCP registry → author a tool → block
   │     LADDER    │   Writes the answer back onto each step.
   └───────┬───────┘
           │  plan, with tools already attached
           ▼
   ┌───────────────┐   A ReAct agent with a 17-tool belt:
   │    BUILDER    │   add_node · validate_workflow · test_workflow · register
   └───────┬───────┘
           │
     ┌─────┴──────┬─────────────┐
     ▼            ▼             ▼
  registered   blocked      gave up
                            (with the last report)
```

**Why split it?** Holding requirements, design, tool-sourcing, wiring, validation *and*
verification in one stream is exactly what a small model drops things from. Planning is a single
question with a single schema-shaped answer — the form a weak model is most reliable at. The
builder then gets told *which tool to use* rather than being asked to remember one.

The source says it plainly:

> *"By the time the builder sees the plan, a step needing a file has `read_file` attached to it.
> The builder is told which tool to use rather than asked to remember one — which is the single
> biggest difference between this working on gpt-4o-mini and not."*
> — `neurosurfer/architect/planner.py`

---

## 3. The plan, on its own

You can run the planner without building anything. This is one model call and it is the cheapest
possible way to see what the Architect *thinks* your request is.

Our intent has a deliberate mix: one step that must reach outside the model, and two that must not.

In [ ]:
INTENT = (
    "Read a text file of customer feedback, pull out the recurring complaints, "
    "and write a short summary for the support lead."
)

from neurosurfer.architect.planner import plan_and_resolve

plan = await plan_and_resolve(provider, INTENT, notify=say)

if not plan.steps:
    # Worth seeing rather than hiding: a small model does occasionally return a
    # plan with no steps. `build()` treats planning as an advantage, not a gate —
    # it notices this and builds without a plan — so §5 works either way.
    print("\n!! the planner returned no steps this run. Re-run this cell to try again;\n"
          "   §5 does not depend on it.")

print(f"\nname:        {plan.name}")
print(f"description: {plan.description}")
print(f"inputs:      {[i.name for i in plan.inputs]}")
print(f"outputs:     {plan.outputs}")

### Reading the plan

The field that matters on every step is **`is_external`** — *does this reach outside the model?*

Reading a file, fetching a URL, calling an API, sending a message: none of that can be done by an
LLM step, however well you word it. A `base` node asked to "read the contents of {path}" does not
read anything — it **invents a plausible file**. That single failure mode is what the whole
capability ladder exists to prevent.

In [ ]:
for s in plan.steps:
    mark = "EXTERNAL" if s.is_external else "internal"
    print(f"[{mark:8}] {s.id}  ({s.kind_hint})")
    print(f"             {s.intent}")
    if s.depends_on:
        print(f"             after: {', '.join(s.depends_on)}")
    if s.is_external:
        print(f"             needs: {s.needed_capability}")
        res = s.resolution or {}
        print(f"             ladder -> status={res.get('status')!r}  tool={res.get('tool')!r}")
    print()

Notice the last two lines on the external step. That `resolution` was **not** produced by the
model — `resolve_plan` walked the ladder in plain Python and wrote the answer back onto the step:

1. search the registered tool catalog,
2. then the MCP registry,
3. then decide: assign it, author a new tool, install a server, or block.

If nothing anywhere provides the capability, the build stops *here* — before a single node exists,
before any prompt is written, and before you have spent a build's worth of tokens on a workflow
that could never have run.

---

## 4. The builder's contract

Before we run it: the builder is a ReAct agent, and its system prompt gives it a toolbelt of 17
tools plus one hard rule about when it is allowed to stop.

| Tool | What it does |
|---|---|
| `set_workflow`, `set_outputs` | name, description, declared inputs, result nodes |
| `add_node`, `update_node`, `remove_node` | build the graph one node at a time |
| `validate_workflow` | run the full rule table — **errors block** |
| `test_workflow` | derive criteria, actually run it, judge the output |
| `find_capability` | search catalog + MCP registry (never guess a tool name) |
| `author_tool`, `install_mcp_server` | acquire a capability that is missing |
| `register_workflow`, `declare_blocked` | the only two ways to finish |

> **"You are finished ONLY after `register_workflow` succeeds or `declare_blocked` is called.
> A turn with only text and no tool call before then is a mistake."**

That last clause is load-bearing. Small models like to end a turn narrating — *"now let me add the
summarise node…"* — which an agent loop reads as a final answer. The Architect detects that and
**nudges** the same conversation back to work rather than restarting it.

---

## 5. Build it

This is the long cell — anywhere from **90 seconds to 8 minutes** on a 9B model. Watch the `·`
lines: you are seeing the plan, then nodes appearing, then validation, then verification.

`build()` plans internally, so it re-runs §3's step for itself. You *can* skip that by passing
`plan=plan` — which is how a plan reviewed in one request gets built by the next one without being
re-invented in between — but note the difference in tolerance:

| | empty / failed plan |
|---|---|
| `build(intent)` | notices, says *"building without a plan"*, carries on |
| `build(intent, plan=p)` | raises `ValueError` — you handed it the plan, so it holds you to it |

Planning is an advantage, not a gate. We take the tolerant path here so this notebook runs even
when a 9B model has an off moment.

In [ ]:
import time
from neurosurfer.architect import ArchitectAgent
from neurosurfer.graph.workflow.registry import WorkflowRegistry

agent = ArchitectAgent(
    provider,
    registry     = WorkflowRegistry(workflows_dir=REGISTRY),
    staging_root = STAGING,
    notify       = say,
    max_turns    = 30,
)

t0 = time.time()
package_path = await agent.build(INTENT)
print(f"\nregistered in {time.time() - t0:.0f}s")
print(package_path)

### What those lines meant

A clean run — the plan was good, so nothing needed repairing:

```
· planned 2 step(s), 1 needing an external capability
·   load_feedback: read a text file from disk → have
· node added: load_feedback (tool) [1/2 steps]
· node added: extract_complaints_summary (base) [2/2 steps]
· validate: ok
· deriving acceptance criteria + test inputs
· verification PASSED (1 graph run; 1 this build)
· REVIEW OK — ...
· Workflow 'summarize_customer_feedback' registered at ...
```

Two things to notice. **`read a text file from disk → have`** is the ladder finding `read_file` in
the catalog, in code, before any node existed. And **`[1/2 steps]`** is the builder tracking the
plan: every planned step must be built or explicitly dropped, so it cannot quietly lose one.

When verification *doesn't* pass first time you get the repair loop instead:

```
· verification FAILED (1 graph run; 1 this build)
· verification FAILED (1 graph run; 2 this build)
· verification PASSED (1 graph run; 3 this build)
```

Each failure is followed by a fix and another real run. The builder is told to **fix the smallest
thing first** — sharpen the failing node's prompt, correct its wiring, swap a tool — and explicitly
*not* to escalate a working router into a loop to patch what is really a prompt bug. The counter
reads `N graph run; M this build`: runs inside *this* verification, then the build's running total.

> This is the expensive half of the Architect, and the part most worth watching. On harder intents
> it is where the time goes.

---

## 6. Read what it wrote

The output is not a black box — it is an ordinary WorkflowPackage on disk. Same format you would
have written by hand in notebook 03.

In [ ]:
from neurosurfer.graph.workflow.package import load_package

pkg = load_package(Path(package_path))

print(f"name:    {pkg.name}")
print(f"inputs:  {[i.name for i in pkg.graph.inputs]}")
print(f"outputs: {pkg.graph.outputs}\n")

for n in pkg.graph.nodes:
    after = f"  after: {', '.join(n.depends_on)}" if n.depends_on else ""
    print(f"  {n.id}  ({n.kind}){after}")
    if n.tools:
        print(f"      tools: {n.tools}")

In [ ]:
# The graph.yaml itself. This is the deliverable — commit it, edit it, run it in CI.
print((Path(package_path) / "graph.yaml").read_text()[:2500])

### The prompts are the interesting part

Look at what it wrote into a node's `purpose` / `goal`, and specifically at the `{placeholders}`.
A node receives **only** what its own text names plus its declared dependencies — there is no
ambient block of every graph input. So an input that appears in no prompt is an input that reaches
no step, which is exactly what §7 is about.

In [ ]:
first = pkg.graph.nodes[0]
for field in ("instructions", "purpose", "goal", "expected_result"):
    val = getattr(first, field, None)
    if val:
        print(f"── {field} ──")
        print(textwrap.indent(textwrap.fill(str(val), 92), "   "), "\n")

---

## 7. The gate: validation

Nothing registers unless `validate_package` passes. The rules are a table, not a wall of `if`s, and
they split into two severities that mean very different things:

- **ERROR** — the workflow will not run, or will not run correctly. **Blocks registration.**
- **WARNING** — it runs, but something is probably not what was meant.

Our built package passes, obviously. The interesting demo is a graph that *looks* fine and isn't.

In [ ]:
from neurosurfer.graph.workflow.validation import validate_package

report = validate_package(pkg)
print(f"ok: {report.ok}   errors: {len(report.errors)}   warnings: {len(report.warnings)}")
if report.issues:
    print(report.summary())

### The rule that catches a workflow which ignores you

Here is a graph that loads, runs green, and is wrong: it **declares** an input and no step ever
names it. A caller passes their article; the model never sees it; the answer is confident and
unrelated.

This used to be a warning, so the workflow registered and the backstop was a human noticing the
answer had nothing to do with what they passed. That is not a backstop a workflow the Architect
builds and verifies *on its own* ever gets — so it blocks now.

In [ ]:
from neurosurfer.graph.engine.schema import Graph, GraphInput, GraphNode
from neurosurfer.graph.workflow.package import WorkflowPackage
from neurosurfer.graph.workflow.schema import WorkflowManifest

def check(node_instructions: str):
    # A one-node graph that declares an `article` input.
    graph = Graph(
        name   = "demo",
        nodes  = [GraphNode(id="a", kind="base", instructions=node_instructions)],
        outputs= ["a"],
        inputs = [GraphInput(name="article", type="string")],
    )
    pkg = WorkflowPackage(manifest=WorkflowManifest(name="demo"), graph=graph, path=Path("."))
    return validate_package(pkg)

# ── the broken one: `article` is declared, and named nowhere ──────────────────
bad = check("Write a summary.")
print(f"registers? {bad.ok}")
for e in bad.errors:
    print(f"  ERROR  {e.message}")
    print(f"         -> {e.suggestion}")

In [ ]:
# ── the fix: name it, and it is read ─────────────────────────────────────────
good = check("Summarise {article}.")
print(f"registers? {good.ok}   (issues: {len(good.issues)})")

The rule counts **every** way a value can be read, not just prompt placeholders — a `map`'s `over`
expression, a `when` guard, `tool_args`, an output node's `value`, a code node's parameter names,
and nodes nested inside container bodies. A rule that only looked at `instructions` would report a
perfectly good fan-out as ignoring its collection.

And it **downgrades itself to a warning where it cannot see**: a `tool` node's arguments live in a
registered schema, and a callable may not import or inspect. Either hides the reads that would
clear the input — and refusing to run over a fact that was never established is worse than the gap.

---

## 8. The proof: verification

Validation asks *"is this well-formed?"*. Verification asks the much harder question:
**"does it actually do what was asked?"** — and answers it by running the thing.

Two calls, and you can drive them yourself:

1. **`derive_acceptance`** — one model call turns the intent + the graph's declared inputs into
   2–6 explicit success criteria, concrete test inputs, and — when the workflow reads a file or a
   directory — a **fixture** that *creates* it.
2. **`verify_workflow`** — runs the staged package on those inputs in a throwaway sandbox, then
   scores each criterion with a judge, **fail-closed** (a criterion the judge doesn't rule on
   counts as failed).

That fixture step is why this is real rather than theatre: a workflow that reads a file cannot be
tested against a *sentence*. A source path with nothing behind it is a hard failure naming the
fixture it needs — not a placeholder string handed to `read_file` so it can fail as "no such file".

> **Heads up:** §5 already verified this workflow successfully, using a rig the agent derived for
> itself. Below we derive a **fresh** rig — and a 9B model sometimes writes a fixture script that
> doesn't parse. If that happens you'll see a FAILED report, and the next section explains why that
> is the system working. Re-running the cell usually produces a script that runs.

In [ ]:
from neurosurfer.architect.agent import derive_acceptance, verify_workflow

declared = [i.model_dump() for i in pkg.graph.inputs]
graph_yaml = (Path(package_path) / "graph.yaml").read_text()

acceptance = await derive_acceptance(provider, INTENT, graph_yaml, declared_inputs=declared)

print("test inputs:", json.dumps(acceptance.test_inputs, indent=2)[:500], "\n")

print("criteria the workflow must satisfy:")
for c in acceptance.criteria:
    print(f"  [{c.id}] {c.description}")

if acceptance.fixtures:
    # A single Fixture: a setup script, and the paths it promises to create.
    print("\nfixture — this workflow reads from disk, so real files get made first:")
    print(f"  creates: {', '.join(acceptance.fixtures.creates)}")

In [ ]:
# Actually run it and judge the result. ~1-2 minutes.
report = await verify_workflow(
    provider,
    intent       = INTENT,
    package_dir  = Path(package_path),
    plan         = acceptance,
    declared_inputs = declared,
)

print(f"PASSED: {report.passed}\n")
print(report.render() if hasattr(report, "render") else report)

### Reading the report — and the distinction that matters

Two very different failures wear the same word, and the report never confuses them:

- **The workflow is wrong.** The run completed and a criterion failed. You get per-criterion
  verdicts plus design suggestions, and the fix belongs in the graph.
- **The test rig is wrong.** The fixture script didn't run, so the workflow never got a fair
  attempt. Then the diagnosis says so in as many words:

  > *"Could not set up the test fixtures — the fixture setup script failed … **This is the test
  > rig, NOT the workflow — do not change the graph, and do not add nodes to create test files.**"*

That last sentence is there because of a real failure mode: told only that "the test file was
missing", the agent started adding `setup_fixtures` / `cleanup_fixtures` **function nodes to the
workflow** — permanently deforming a correct design to satisfy a broken harness. Naming which side
is at fault is what stops that.

And the scoring is **fail-closed**: a criterion the judge doesn't rule on counts as failed. A
verification that cannot reach a verdict is a verification that did not pass — never a pass by
default.

---

## 9. When it refuses

The most important behaviour in the whole subsystem is the one that produces *nothing*.

Ask for something that needs credentials or resources you never supplied, and the Architect calls
`declare_blocked` — which surfaces as `WorkflowInfeasible` carrying a checklist of exactly what is
missing. A clear blocker beats a workflow that pretends.

Note what is **not** a blocker: needing to analyse, summarise, classify or write is never a reason
to refuse. That is what a `base` node is for. The refusal is reserved for capabilities that
genuinely are not available.

In [ ]:
from neurosurfer.architect import WorkflowInfeasible

blocked_agent = ArchitectAgent(
    provider,
    registry     = WorkflowRegistry(workflows_dir=REGISTRY),
    staging_root = STAGING / "blocked",
    notify       = say,
    max_turns    = 20,
)

try:
    await blocked_agent.build(
        "Build a workflow that logs into my company's production Oracle database "
        "and deletes duplicate customer rows."
    )
    print("!! it built something — that is a finding, not a pass")
except WorkflowInfeasible as e:
    print("\nBLOCKED, as it should be:\n")
    print(textwrap.indent(textwrap.fill(str(e), 92), "  "))

### What a good refusal looks like

Notice how far it got before refusing. It planned three steps, resolved two of them to the `sql`
tool, built all three nodes — and *then* blocked, because validation showed the capability it had
been given could not do the job:

> *"the `sql` tool in the catalog is **read-only** and cannot perform DELETE operations … The
> available capabilities are: test_connection, list_tables, table_schema, query. To build this
> workflow, either (1) `author_tool` to create a Python tool that can execute write operations, or
> (2) install an MCP server that provides read-write database access."*

That is the shape to want: it names the **specific** gap (write access, not "database access"),
lists what it *does* have, and gives two concrete routes forward. Compare it to the alternative —
a registered workflow with a `remove_duplicates` node that quietly does nothing, and a green run
telling you your duplicates are gone.

Also worth noting: it did **not** block on the analysis steps. Needing to summarise, classify or
write is never a reason to refuse — that is what a `base` node is for. Only a genuinely absent
capability blocks.

---

## 10. Where this runs from

Three doors into the same Architect:

| Door | How |
|---|---|
| **Python** | `ArchitectAgent(provider).build(intent)` — what this notebook used |
| **Studio** | the build stream renders the staged graph live as nodes appear |
| **CLI** | `neurosurfer workflow create` |

> ⚠️ One inconsistency worth knowing: the CLI currently routes to `ArchitectBuilder`, an **older
> fixed 8-node pipeline** (`discover → clarify → decompose → design_nodes → critique →
> tool_design → write_nodes → assemble`) that still ships in `neurosurfer/architect/package/`.
> `ArchitectAgent` — the ReAct agent this notebook drove — is the current path and what the studio
> uses. Prefer the Python or studio route until the CLI is moved over.

Once registered, a workflow is just a package. Run it like any other:

```python
from neurosurfer.graph.workflow.runner import WorkflowRunner
result = WorkflowRunner(provider).run(pkg, {"feedback_file": "feedback.txt"})
```

---

## Summary

You handed an agent a sentence and got back a runnable, validated, **tested** workflow package.

| Stage | Call | What it guarantees |
|---|---|---|
| Plan | `plan_and_resolve` | one structured answer — the form a weak model is reliable at |
| Ladder | `resolve_capability` | tools chosen **in code**, not remembered by the model |
| Build | `ArchitectAgent.build` | one node at a time, warnings read after every call |
| Gate | `validate_package` | errors block; an ignored parameter is now one of them |
| Proof | `derive_acceptance` + `verify_workflow` | it ran, on a real fixture, judged fail-closed |
| Refusal | `WorkflowInfeasible` | a checklist, not a workflow that pretends |

### The three ideas worth taking away

1. **Split the hard question from the long one.** Planning is one schema-shaped call; building is a
   loop. Merging them is what makes small models drop requirements.

2. **Do the deterministic part deterministically.** The capability ladder is plain Python. Asking a
   model to remember tool names is the most reliable way to get invented ones.

3. **A workflow that runs green is not a workflow that works.** Structure validation and behaviour
   verification are different questions, and the Architect is only trustworthy because it asks
   both — then repairs what it broke and asks again.

### Where to go next

- **[03 — Graph Agents](03_graph_agents.ipynb)** — the format the Architect emits, by hand.
- **[05 — Capstone](05_capstone_insight_engine.ipynb)** — a seven-node graph a human designed.
- Open `tutorials/tmp/architect/registry/` and read the `graph.yaml` it wrote. Editing it by hand
  is the fastest way to understand what it decided, and why.